# Formulaire — Probabilistic Modeling & Bayesian Reasoning

> **Mode d'emploi** : remplace uniquement les variables dans les blocs `# === VARIABLES À CHANGER ===`. Le reste du code tourne tel quel.

---

## EXERCICE 1 — Simulation & variabilité

### Formules clés

| Grandeur | Formule |
|---|---|
| Fréquence empirique | `freq = data.mean()` |
| Écart-type théorique | $\sigma = \sqrt{\dfrac{p(1-p)}{n}}$ |
| Biais de l'estimateur | $\mathbb{E}[\hat{p}] = p$ → **sans biais** |

### Ce qui change d'un exam à l'autre
- `p_anomaly` : probabilité de l'événement rare
- `n` : taille d'un échantillon
- `n_simulations` : nombre de répétitions (souvent 2500)
- Les tailles à comparer dans Q2 (ex: 50 et 800)

### Interprétation automatique
- Std **grande** → estimateur peu fiable, échantillon petit
- Std **petite** → estimateur précis, grand n
- Moyenne des simulations ≈ p_réel → estimateur **sans biais**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === VARIABLES À CHANGER ===
SEED         = 123
P_ANOMALY    = 0.12   # probabilité de l'événement
N            = 150    # taille d'un échantillon
N_SIMUL      = 2500   # nombre de simulations
N_COMPARE    = [50, 800]  # tailles à comparer pour Q2
# ===========================

np.random.seed(SEED)
data = np.random.binomial(1, P_ANOMALY, size=N)
freq_emp = data.mean()

freqs = np.array([np.random.binomial(1, P_ANOMALY, size=N).mean()
                  for _ in range(N_SIMUL)])

plt.figure(figsize=(8, 4))
plt.hist(freqs, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(P_ANOMALY, color='red',    linestyle='--', label=f'p réel = {P_ANOMALY}')
plt.axvline(freq_emp,  color='orange', linestyle='--', label=f'freq observée = {freq_emp:.3f}')
plt.xlabel("Fréquence"); plt.ylabel("Nombre de simulations")
plt.title(f"Distribution des fréquences (n={N}, {N_SIMUL} simulations)")
plt.legend(); plt.tight_layout(); plt.show()

print(f"Fréquence empirique      : {freq_emp:.4f}")
print(f"Moyenne des simulations  : {freqs.mean():.4f}  (attendu ≈ {P_ANOMALY})")
print(f"Écart-type simulé        : {freqs.std():.4f}")
print(f"Écart-type théorique     : {np.sqrt(P_ANOMALY*(1-P_ANOMALY)/N):.4f}")

print()
for n_test in N_COMPARE:
    f = np.array([np.random.binomial(1, P_ANOMALY, size=n_test).mean()
                  for _ in range(N_SIMUL)])
    sigma_th = np.sqrt(P_ANOMALY*(1-P_ANOMALY)/n_test)
    print(f"n={n_test:>4} → mean={f.mean():.4f}  std_simulé={f.std():.4f}  std_théorique={sigma_th:.4f}")

---
## EXERCICE 2 — Hypothèses discrètes

### Formules clés

$$P(\text{alerte} \mid H) = p_{\text{intrusion}} \times \text{detect\_rate}_H + (1 - p_{\text{intrusion}}) \times p_{\text{faux positif}}$$

**Bayes discret (prior 50/50) :**
$$P(H=1 \mid \text{data}) = \frac{\mathcal{L}(H=1)}{\mathcal{L}(H=0) + \mathcal{L}(H=1)}$$

où $\mathcal{L}(H) = \text{Binomial}(k \mid n, p_{\text{alerte} \mid H})$

### Ce qui change d'un exam à l'autre
- `P_INTRUSION` : probabilité d'une vraie intrusion
- `DETECT_H0` / `DETECT_H1` : taux de détection selon l'état du système
- `P_FAUX_POS` : taux de faux positifs
- `N` : nombre d'observations

### Interprétation
- P(H=1|data) **proche de 0.5** → les deux hypothèses sont indiscernables (taux d'alertes proches)
- P(H=1|data) **proche de 0 ou 1** → les données permettent de trancher
- La différence `p_alert_H0 - p_alert_H1` quantifie la **discriminabilité** du test

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
from scipy.stats import binom

# === VARIABLES À CHANGER ===
SEED         = 7
N            = 400
P_INTRUSION  = 0.10
DETECT_H0    = 0.80   # taux détection système NORMAL
DETECT_H1    = 0.55   # taux détection système COMPROMIS
P_FAUX_POS   = 0.15   # prob alerte sans intrusion (identique pour H0 et H1)
# ===========================

np.random.seed(SEED)
intrusion = np.random.binomial(1, P_INTRUSION, size=N)
alerts = np.where(
    intrusion == 1,
    np.random.binomial(1, DETECT_H0, size=N),
    np.random.binomial(1, P_FAUX_POS, size=N)
)

p_alert_H0 = P_INTRUSION * DETECT_H0 + (1 - P_INTRUSION) * P_FAUX_POS
p_alert_H1 = P_INTRUSION * DETECT_H1 + (1 - P_INTRUSION) * P_FAUX_POS
n_alerts   = alerts.sum()

print(f"Alertes observées        : {n_alerts} / {N}")
print(f"P(alerte | H=0, normal)  = {p_alert_H0:.4f}")
print(f"P(alerte | H=1, compromis) = {p_alert_H1:.4f}")
print(f"Différence               = {abs(p_alert_H0 - p_alert_H1):.4f}")

# Analytique (vérification rapide avant PyMC)
lik_H0 = binom.pmf(n_alerts, N, p_alert_H0)
lik_H1 = binom.pmf(n_alerts, N, p_alert_H1)
p_H1_anal = lik_H1 / (lik_H0 + lik_H1)
print(f"\nP(H=1 | données) analytique = {p_H1_anal:.4f}")

# PyMC
with pm.Model() as model_cyber:
    H       = pm.Bernoulli("H", p=0.5)
    p_alert = pm.math.switch(pm.math.eq(H, 1), p_alert_H1, p_alert_H0)
    obs     = pm.Binomial("obs", n=N, p=p_alert, observed=n_alerts)
    trace   = pm.sample(4000, tune=1000, chains=2, progressbar=True,
                        idata_kwargs={"log_likelihood": True})

p_H1_mcmc = float(trace.posterior["H"].values.mean())
print(f"P(H=1 | données) via PyMC   = {p_H1_mcmc:.4f}")
az.plot_posterior(trace, var_names=["H"])

---
## EXERCICE 3 — Inférence Bayésienne & décision

### Formules clés

**Mise à jour conjuguée Beta-Binomiale :**
$$\text{Prior } \text{Beta}(\alpha, \beta) \;+\; k \text{ succès sur } n \;\Rightarrow\; \text{Postérieur } \text{Beta}(\alpha+k,\; \beta+n-k)$$

| Grandeur | Formule |
|---|---|
| Moyenne a posteriori | $\mu = \dfrac{\alpha_{\text{post}}}{\alpha_{\text{post}} + \beta_{\text{post}}}$ |
| Influence du prior | $\dfrac{\alpha+\beta}{\alpha+\beta+n}$ |
| Prior non-informatif | $\text{Beta}(1, 1)$ |

### Priors courants

| Prior | Signification |
|---|---|
| Beta(1, 1) | Non-informatif (uniforme) |
| Beta(2, 2) | Légèrement centré sur 0.5 |
| Beta(6, 2) | Optimiste (p élevé probable) |
| Beta(2, 6) | Pessimiste (p faible probable) |

### Ce qui change d'un exam à l'autre
- `N_OBS` : nombre d'observations
- `K_SUCCESS` : nombre de succès
- `ALPHA_PRIOR` / `BETA_PRIOR` : paramètres du prior
- `ALPHA_OPT` / `BETA_OPT` : prior alternatif (Q2)
- `SEUIL_*` : seuils de la règle de décision

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# === VARIABLES À CHANGER ===
N_OBS        = 20     # nombre d'observations
K_SUCCESS    = 13     # nombre de succès
ALPHA_PRIOR  = 1      # prior non-informatif
BETA_PRIOR   = 1
ALPHA_OPT    = 6      # prior optimiste (Q2)
BETA_OPT     = 2
SEUIL_DEPLOY = 0.75   # au-dessus → déploiement
SEUIL_ABANDON= 0.50   # en-dessous → abandon
SEUIL_CALCUL = 0.65   # seuil pour P(p > x) demandé
# ===========================

K_FAIL = N_OBS - K_SUCCESS

# --- Prior non-informatif ---
a_post = ALPHA_PRIOR + K_SUCCESS
b_post = BETA_PRIOR  + K_FAIL
post   = stats.beta(a_post, b_post)
mu     = post.mean()

print(f"Postérieur (non-inf)  : Beta({a_post}, {b_post})")
print(f"Moyenne a posteriori  : {mu:.4f}")
print(f"P(p > {SEUIL_CALCUL}) : {1 - post.cdf(SEUIL_CALCUL):.4f}")
print(f"P(p > {SEUIL_DEPLOY}) : {1 - post.cdf(SEUIL_DEPLOY):.4f}")

if mu >= SEUIL_DEPLOY:
    decision = "DÉPLOIEMENT"
elif mu >= SEUIL_ABANDON:
    decision = "TEST SUPPLÉMENTAIRE"
else:
    decision = "ABANDON"
print(f"Décision              : {decision}  (moyenne {mu:.3f})")

# --- Prior optimiste ---
a_opt = ALPHA_OPT + K_SUCCESS
b_opt = BETA_OPT  + K_FAIL
post_opt = stats.beta(a_opt, b_opt)
mu_opt   = post_opt.mean()
inf_prior = (ALPHA_OPT + BETA_OPT) / (ALPHA_OPT + BETA_OPT + N_OBS)

print(f"\nPostérieur (optimiste) : Beta({a_opt}, {b_opt})")
print(f"Moyenne a posteriori   : {mu_opt:.4f}")
print(f"P(p > {SEUIL_DEPLOY})  : {1 - post_opt.cdf(SEUIL_DEPLOY):.4f}")
print(f"Influence du prior     : {inf_prior:.1%}  ({ALPHA_OPT+BETA_OPT} pseudo-obs / {ALPHA_OPT+BETA_OPT+N_OBS} total)")

# Visualisation
x = np.linspace(0, 1, 500)
plt.figure(figsize=(9, 4))
plt.plot(x, post.pdf(x),     label=f"Prior Beta({ALPHA_PRIOR},{BETA_PRIOR}) → Post Beta({a_post},{b_post})", color='steelblue')
plt.plot(x, post_opt.pdf(x), label=f"Prior Beta({ALPHA_OPT},{BETA_OPT}) → Post Beta({a_opt},{b_opt})",     color='orange')
plt.axvline(SEUIL_CALCUL, color='gray', linestyle=':',  label=f'seuil {SEUIL_CALCUL}')
plt.axvline(SEUIL_DEPLOY, color='red',  linestyle='--', label=f'seuil déploiement {SEUIL_DEPLOY}')
plt.xlabel("p"); plt.ylabel("Densité")
plt.title("Distributions a posteriori")
plt.legend(); plt.tight_layout(); plt.show()

---
## EXERCICE 4 — Comparaison de modèles & PPC

### Formules clés

**Moyenne de référence pour M1 :**
$$\mu_{M1} = \frac{1}{n}\sum x_i$$

**LOO (Leave-One-Out cross-validation) :**
- Score **plus élevé** (moins négatif) = modèle **meilleur**
- `elpd_diff` : différence de score — si grande et `dse` petite → différence significative

**Posterior Predictive Check (PPC) :**
- Simule de nouvelles données depuis le postérieur
- Un bon modèle reproduit la **forme** des vraies données

### Ce qui change d'un exam à l'autre
- `DATA` : tableau des observations
- `GROUP` : tableau 0/1 indiquant le régime de chaque obs
- Priors sur `mu`, `mu0`, `mu1`, `sigma` (à adapter à l'échelle des données)

### Règle rapide pour choisir les priors
- Centrer `mu` sur la moyenne globale des données
- Centrer `mu0` / `mu1` sur la moyenne de chaque groupe visible
- `sigma` : HalfNormal avec sigma ≈ écart-type intra-groupe attendu

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

# === VARIABLES À CHANGER ===
DATA  = np.array([5.1, 5.3, 5.0, 5.2, 9.8, 10.1, 9.9, 10.2])
GROUP = np.array([0,   0,   0,   0,   1,   1,    1,   1   ])
# Priors M1
MU_PRIOR_M1    = DATA.mean()  # centré sur la moyenne globale
SIGMA_PRIOR_M1 = 5
# Priors M2
MU0_PRIOR = DATA[GROUP==0].mean()  # centré sur chaque groupe
MU1_PRIOR = DATA[GROUP==1].mean()
SIGMA_MU   = 2
SIGMA_OBS  = 1
# ===========================

print(f"Moyenne globale (ref M1) : {DATA.mean():.4f}")
print(f"Moyenne groupe 0        : {DATA[GROUP==0].mean():.4f}")
print(f"Moyenne groupe 1        : {DATA[GROUP==1].mean():.4f}")

# M1 : une seule moyenne
with pm.Model() as M1:
    mu    = pm.Normal("mu", mu=MU_PRIOR_M1, sigma=SIGMA_PRIOR_M1)
    sigma = pm.HalfNormal("sigma", sigma=SIGMA_PRIOR_M1)
    obs   = pm.Normal("obs", mu=mu, sigma=sigma, observed=DATA)
    tr1   = pm.sample(2000, tune=1000, chains=2, progressbar=False,
                      idata_kwargs={"log_likelihood": True})
    ppc1  = pm.sample_posterior_predictive(tr1)

# M2 : deux régimes
with pm.Model() as M2:
    mu0   = pm.Normal("mu0", mu=MU0_PRIOR, sigma=SIGMA_MU)
    mu1   = pm.Normal("mu1", mu=MU1_PRIOR, sigma=SIGMA_MU)
    sigma = pm.HalfNormal("sigma", sigma=SIGMA_OBS)
    mu    = pm.math.switch(pm.math.eq(GROUP, 0), mu0, mu1)
    obs   = pm.Normal("obs", mu=mu, sigma=sigma, observed=DATA)
    tr2   = pm.sample(2000, tune=1000, chains=2, progressbar=False,
                      idata_kwargs={"log_likelihood": True})
    ppc2  = pm.sample_posterior_predictive(tr2)

# LOO
comp = az.compare({"M1": tr1, "M2": tr2}, ic="loo")
print("\n--- Comparaison LOO ---")
print(comp)
print("→ Meilleur modèle :", comp.index[0])

# PPC
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, ppc, name in zip(axes, [ppc1, ppc2], ["M1 (1 régime)", "M2 (2 régimes)"]):
    vals = ppc.posterior_predictive["obs"].values.reshape(-1, len(DATA))
    for i in range(min(200, len(vals))):
        ax.scatter(range(len(DATA)), vals[i], alpha=0.02, color='steelblue', s=10)
    ax.scatter(range(len(DATA)), DATA, color='red', zorder=5, label='Observations')
    ax.set_title(f"PPC — {name}"); ax.legend()
plt.tight_layout(); plt.show()

---
## Aide-mémoire rapide

### Règles d'interprétation universelles

| Situation | Ce qu'il faut dire |
|---|---|
| Std simulée ≈ théorique | Simulation correcte |
| Std diminue quand n augmente | Loi des grands nombres |
| P(H=1\|data) ≈ 0.5 | Hypothèses indiscernables (taux proches) |
| P(H=1\|data) ≈ 0 ou 1 | Données discriminantes |
| Prior influence ++ | n petit ou prior informatif |
| LOO M2 > LOO M1 | M2 prédit mieux, mais vérifier PPC |
| PPC concentré sur les données | Modèle crédible |

### Formule Beta conjuguée (à mémoriser)

$$\boxed{\text{Beta}(\alpha, \beta) + (k \text{ succès}, n{-}k \text{ échecs}) \;\Rightarrow\; \text{Beta}(\alpha+k,\; \beta+n-k)}$$

### Influence du prior

$$\boxed{\text{Influence prior} = \frac{\alpha+\beta}{\alpha+\beta+n}}$$

Plus $n$ est grand, plus cette fraction tend vers 0 → les données prennent le dessus.